# SIPTA -- Validación: Demografía y Población (Localidad & UPL)
**Proyecto**: Sistema de Indicadores y Priorización Territorial y Alertas Tempranas (DataJam Bogotá)  
**Fase PDCO**: CONTROL | **Fase CRISP-DM**: Data Quality & Validation  
**Autoría**: Persona A (Adan Sánchez) & Persona B (Yesid Bello)  
**Fuente**: `data/raw/DEMOGRAFIA/osb_demografia-poblacion-localidad.csv` y `osb_demografia-poblacion-upl.csv`  
**Temporalidad**: **2005 - 2035 (Proyecciones Anuales SDP - DANE, CNPV 2018)**  
**Indicadores Habilitados**: `DEM-001` (Densidad Poblacional), `POB-001..004` (Población por Grupos de Edad)


## 1. Validación de Calidad Técnica con `src/validation/validate_data.py`


In [ ]:
import sys
from pathlib import Path

# Resolver la raíz del proyecto SIPTA
for p in [Path('.').resolve(), Path('.').resolve().parent, Path('.').resolve().parent.parent]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd
from src.validation.validate_data import validate_demografia, inspect_schema, load_raw_demografia

report = validate_demografia()
print("=== REPORTE EJECUTIVO DE VALIDACIÓN ===")
print(f"Dominio: {report['domain']}")
print(f"Temporalidad Oficial: {report['temporalidad']}")
print(f"Total Registros: {report['total_rows']:,}")
print(f"Rango de Años: {report['years_covered'][0]} a {report['years_covered'][-1]}")
print(f"Estado de Calidad: {report['validation_status']}")



## 2. Inspección Detallada de Esquema y Nulos


In [ ]:
df_loc, df_upl = load_raw_demografia()
schema_df = inspect_schema(df_loc)
display(schema_df)



## 3. Demostración de Cálculo de Indicadores (`DEM-001` y `POB-002`)


In [ ]:
# 1. Poblacion oficial DANE 2025 por Localidad
if 'ano' in df_loc.columns and 'area' in df_loc.columns:
    df_2025 = df_loc[(df_loc['ano'] == 2025) & (df_loc['area'] == 'Total')].copy()
else:
    df_2025 = df_loc.copy()
col_pob = 'poblacion_total' if 'poblacion_total' in df_2025.columns else ('poblacion_2025' if 'poblacion_2025' in df_2025.columns else df_2025.columns[-1])
print('Demostracion de denominadores demograficos oficiales DANE / SDP (2025):')
print(f'Total Poblacion Bogota 2025: {df_2025[col_pob].sum():,}')
display(df_2025.head(10))


## 4. Dictamen de Validez
- **Fuente Válida**: Sí. Cubre el 100% de las 20 localidades oficiales sin nulos en `CODIGO_LOCALIDAD` ni `POBLACION`.
- **Uso Metodológico**: Denominador per cápita oficial de todo el sistema SIPTA.
